In [ ]:
from metabolic_shortest_distance import MetabolicShortestDistance
from essential.fba import load_ecoli_rich_medium_model
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotnine as gg
from fba_utils import to_long_no_diagonal
import scipy.stats as stats
from sklearn.decomposition import PCA


SHARED_THEME = gg.theme(
    axis_text=gg.element_text(size=6),
    axis_title=gg.element_text(size=7),
    figure_size=(3, 2),
    title=gg.element_text(size=7),
    legend_text=gg.element_text(size=6),
)

In [ ]:
model = load_ecoli_rich_medium_model()
currency = [
    "atp_c",
    "atp_e",
    "adp_c",
    "adp_e",
    "h2o_c",
    "h2o_e",
    "h_c",
    "h_e",
    "nad_c",
    "nadh_c",
    "nadp_c",
    "nadph_c",
    "coa_c",
    "pi_c",
    "pi_e",
    "ppi_c",
]
msd = MetabolicShortestDistance(model, currency)

# %%
print("Graph size:", msd.adj_matrix.shape)
print("Computing distance between first two genes:")
d = msd.compute_distance(msd.genes[0], msd.genes[1])
print(f"d({msd.genes[0]} -> {msd.genes[1]}) =", d)

print("Computing all distances...")

In [ ]:
metabolic_df = msd.compute_all_distances()
metabolic_df = metabolic_df.merge(
    metabolic_df,
    left_on=["gene_start", "gene_stop"],
    right_on=["gene_stop", "gene_start"],
    suffixes=["", "_reverse"],
    how="left",
)
metabolic_df.loc[lambda x: x.gene_start == x.gene_stop, "directed_distance"] = 0
metabolic_df.loc[:, "distance"] = np.minimum(
    metabolic_df.directed_distance, metabolic_df.directed_distance_reverse
)

In [ ]:
def print_substrates_products(gene):
    substrates = [msd.idx_to_metabolite[x] for x in msd.gene_substrates[gene]]
    products = [msd.idx_to_metabolite[x] for x in msd.gene_products[gene]]
    print(f"{gene} substrates: {', '.join(substrates)}")
    print(f"{gene} products: {', '.join(products)}")

In [ ]:
df_wide = metabolic_df.pivot(index="gene_start", columns="gene_stop", values="directed_distance")
df_wide_undirected = np.minimum(df_wide.values, df_wide.values.T)
df_wide_undirected = df_wide.clip(upper=20)

In [ ]:
df_wide_undirected.stack().plot.hist(bins=100)

In [ ]:
sns.clustermap(df_wide_undirected, cmap="rocket_r", vmax=2)

In [ ]:
from sklearn.manifold import TSNE

tsne = TSNE(n_components=2, random_state=42, metric="precomputed", init="random")
tsne_results = tsne.fit_transform(df_wide_undirected)
df_wide_undirected_tsne = pd.DataFrame(
    tsne_results, index=df_wide_undirected.index, columns=["TSNE1", "TSNE2"]
).assign(gene_name=lambda x: x.index)

In [ ]:
import plotly.express as px


fig = px.scatter(
    df_wide_undirected_tsne,
    x="TSNE1",
    y="TSNE2",
    hover_name="gene_name",
    template="plotly_white",
    width=600,
    height=400,
)
fig.update_traces(marker=dict(size=3))
fig.show()

In [ ]:
gene_subset = ["valS", "ileS", "livG", "livF", "livM", "leuS", "tdh", "thrS", "livJ", "livH"]
df_wise_subset = df_wide_undirected.loc[gene_subset].loc[:, gene_subset]
sns.clustermap(df_wise_subset, cmap="rocket_r")

In [ ]:
gene_subset = ["menA", "menB", "ybgC", "menH"]
df_wise_subset = df_wide_undirected.loc[gene_subset].loc[:, gene_subset]
sns.clustermap(df_wise_subset, cmap="rocket_r")

In [ ]:
from fba_utils import compute_pairwise
import scanpy as sc
from tqdm import tqdm

adata = sc.read_h5ad(
    "/workspace/data/251117_genomescale_CRISPRi/sample_mix_umi200_hvg500_pc25_neighbors10_mindist0.55.scvi.h5ad"
)
adata_case = sc.read_h5ad("/workspace/data/251117_genomescale_CRISPRi/adata_case.annotated.h5ad")

transcript_df = []
transcript_case_df = []
z_transcript_df = []
z_transcript_case_df = []
gene_names = []
gene_case_names = []


for gene in tqdm(adata.obs["gene"].unique()):
    adata_gene = adata[adata.obs["gene"] == gene]
    X_gene = adata_gene.layers["cp10k"].toarray()
    if X_gene.shape[0] > 0:
        gene_names.append(gene)
        transcript_df.append(X_gene.mean(axis=0))
        z_transcript_df.append(adata_gene.obsm["X_scVI"].mean(axis=0))

    adata_gene_case = adata_case[adata_case.obs["gene"] == gene]
    X_gene_case = adata_gene_case.layers["cp10k"].toarray()
    if X_gene_case.shape[0] > 0:
        gene_case_names.append(gene)
        transcript_case_df.append(X_gene_case.mean(axis=0))
        z_transcript_case_df.append(adata_gene_case.obsm["X_scVI"].mean(axis=0))
transcript_df = pd.DataFrame(transcript_df, index=gene_names)
transcript_case_df = pd.DataFrame(transcript_case_df, index=gene_case_names)
z_transcript_df = pd.DataFrame(z_transcript_df, index=gene_names)
z_transcript_case_df = pd.DataFrame(z_transcript_case_df, index=gene_case_names)

In [ ]:
transcript_df_pca = PCA(n_components=50).fit_transform(transcript_df)
transcript_df_pca_ = pd.DataFrame(transcript_df_pca, index=gene_names)
transcript_pairwise = compute_pairwise(transcript_df_pca_, metric="euclidean")
distance_pc = to_long_no_diagonal(transcript_pairwise).rename(columns={"distance": "distance_pc"})

transcript_df_pca = PCA(n_components=50).fit_transform(transcript_case_df)
transcript_df_pca_ = pd.DataFrame(transcript_df_pca, index=gene_case_names)
transcript_pairwise = compute_pairwise(transcript_df_pca_, metric="euclidean")
distance_pc_case = to_long_no_diagonal(transcript_pairwise).rename(
    columns={"distance": "distance_pc_case"}
)

transcript_pairwise = compute_pairwise(z_transcript_case_df, metric="euclidean")
distance_z_case = to_long_no_diagonal(transcript_pairwise).rename(
    columns={"distance": "distance_z_case"}
)

transcript_distances = distance_pc.merge(distance_pc_case, on=["gene1", "gene2"]).merge(
    distance_z_case, on=["gene1", "gene2"]
)

In [ ]:
joint_dists = metabolic_df.assign(distance_metabolic=lambda x: np.clip(x.distance, 0, 20)).merge(
    transcript_distances,
    left_on=["gene_start", "gene_stop"],
    right_on=["gene1", "gene2"],
)

In [ ]:
def compare_distances(transcript_distance_name):
    joint_dists_ = joint_dists.copy().loc[lambda x: x.distance_metabolic <= 11]

    corr_ = stats.spearmanr(
        joint_dists_["distance_metabolic"], joint_dists_[transcript_distance_name]
    )[0]
    
    fig = (
        gg.ggplot(joint_dists_, gg.aes(x="factor(distance_metabolic)", y=transcript_distance_name))
        # + gg.geom_point(size=0.5)
        + gg.geom_boxplot()
        + gg.theme_minimal()
        + gg.labs(x="Metabolic distance", title=f"Spearman rho = {corr_:.2f}")
        + SHARED_THEME
        + gg.theme(figure_size=(3, 2))
    )
    display(fig)

    mean_comparison = (
        joint_dists_.groupby("distance_metabolic")[transcript_distance_name].mean().reset_index()
    )
    spearman_comparison = stats.spearmanr(
        mean_comparison["distance_metabolic"], mean_comparison[transcript_distance_name]
    )
    fig = (
        gg.ggplot(mean_comparison, gg.aes(x="distance_metabolic", y=transcript_distance_name))
        + gg.geom_point(size=0.5)
        + gg.theme_minimal()
        + gg.labs(
            x="Metabolic distance",
            y=f"Mean {transcript_distance_name}",
            title=f"Spearman rho = {spearman_comparison[0]:.2f}",
        )
        + SHARED_THEME
        + gg.theme(figure_size=(3, 2))
    )
    display(fig)

In [ ]:
fig = compare_distances("distance_pc_case")
fig = compare_distances("distance_pc")
fig = compare_distances("distance_z_case")

In [ ]:
(
    joint_dists.loc[lambda x: x.distance_metabolic == 0]
    .sort_values("distance_pc_case", ascending=False)
    .head(10)
)

In [ ]:
print_substrates_products("metK")
print_substrates_products("bioB")